# v29 Full Next Session

If v29_standard_summary exists and selected calibrator is safe, reconstruct output.
Otherwise rollback to base v27.

In [ ]:
import os, re, json, glob, random, math, statistics
from pathlib import Path
from collections import Counter, defaultdict

LABELS = ["A","B","C","D","Yes","No","Unknown"]
MC_LABELS = {"A","B","C","D"}
YNU_LABELS = {"Yes","No","Unknown"}
OUT_DIR = Path("/kaggle/working/v29_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)

def norm_answer(x):
    if x is None: return "UNPARSEABLE"
    s = str(x).strip()
    if not s: return "UNPARSEABLE"
    low = s.lower()
    if low in ["a","option a","answer a"]: return "A"
    if low in ["b","option b","answer b"]: return "B"
    if low in ["c","option c","answer c"]: return "C"
    if low in ["d","option d","answer d"]: return "D"
    if low in ["yes","true","entailed","supported"]: return "Yes"
    if low in ["no","false","not entailed","unsupported","contradicted"]: return "No"
    if low in ["unknown","uncertain","not enough information","cannot determine"]: return "Unknown"
    m = re.search(r"\b(final answer|answer)\s*[:\-]\s*(A|B|C|D|Yes|No|Unknown)\b", s, flags=re.I)
    if m: return norm_answer(m.group(2))
    m = re.search(r"\b(A|B|C|D|Yes|No|Unknown)\b", s, flags=re.I)
    if m: return norm_answer(m.group(1))
    return "UNPARSEABLE"

def to_jsonable(x):
    try:
        import numpy as np
        if isinstance(x, (np.integer,)): return int(x)
        if isinstance(x, (np.floating,)): return float(x)
        if isinstance(x, (np.bool_,)): return bool(x)
    except Exception:
        pass
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)
    print("Saved:", path)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_one(patterns, required=True, label="file"):
    hits = []
    for pat in patterns:
        hits += glob.glob(pat, recursive=True)
    hits = sorted(set(hits))
    print(f"{label} candidates:", hits[:20], "..." if len(hits) > 20 else "")
    if not hits:
        if required:
            raise FileNotFoundError(f"Missing {label}; checked {patterns}")
        return None
    hits = sorted(hits, key=lambda p: (
        0 if "/kaggle/input/" in p else 1,
        0 if "v27_standard_preds" in Path(p).name else 1,
        1 if "ORACLE" in Path(p).name.upper() else 0,
        len(p)
    ))
    print("Using", label, ":", hits[0])
    return Path(hits[0])

def compute_metrics(rows, name="metric"):
    y_true = [norm_answer(r.get("gold")) for r in rows]
    y_pred = [norm_answer(r.get("pred")) for r in rows]
    n = len(rows)
    acc = sum(t == p for t,p in zip(y_true,y_pred)) / max(1,n)
    per, f1s, weights = {}, [], []
    for lab in LABELS:
        tp = sum(t == lab and p == lab for t,p in zip(y_true,y_pred))
        fp = sum(t != lab and p == lab for t,p in zip(y_true,y_pred))
        fn = sum(t == lab and p != lab for t,p in zip(y_true,y_pred))
        support = sum(t == lab for t in y_true)
        pred_count = sum(p == lab for p in y_pred)
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        per[lab] = dict(precision=precision, recall=recall, f1=f1, support=support, pred_count=pred_count, tp=tp, fp=fp, fn=fn)
        f1s.append(f1); weights.append(support)
    return dict(
        name=name, n=n, acc=acc, macro_f1=sum(f1s)/len(f1s),
        weighted_f1=sum(f*w for f,w in zip(f1s,weights))/max(1,sum(weights)),
        mc_macro_f1=sum(per[x]["f1"] for x in ["A","B","C","D"])/4,
        ynu_macro_f1=sum(per[x]["f1"] for x in ["Yes","No","Unknown"])/3,
        per_label=per,
    )

def print_metric(m):
    print("="*100)
    print(m["name"])
    print("="*100)
    print(f"N={m['n']} acc={m['acc']:.4f} macro={m['macro_f1']:.4f} weighted={m['weighted_f1']:.4f}")
    print(f"MC={m['mc_macro_f1']:.4f} YNU={m['ynu_macro_f1']:.4f}")
    print("-"*100)
    print(f"{'Label':<10} {'P':>8} {'R':>8} {'F1':>8} {'Gold':>7} {'Pred':>7} {'Flag':>14}")
    for lab in LABELS:
        d = m["per_label"][lab]
        flag = "OK"
        if d["pred_count"] == 0 and d["support"] > 0: flag = "COLLAPSE"
        elif d["support"] and d["recall"] < 0.25: flag = "LOW_RECALL"
        elif d["support"] and d["f1"] < 0.35: flag = "LOW_F1"
        print(f"{lab:<10} {d['precision']:8.3f} {d['recall']:8.3f} {d['f1']:8.3f} {d['support']:7d} {d['pred_count']:7d} {flag:>14}")

def load_base_preds():
    p = resolve_one([
        "/kaggle/input/**/v27_standard_preds.json",
        "/kaggle/working/**/v27_standard_preds.json",
        "/kaggle/input/**/v25_operational_preds*.json",
        "/kaggle/working/**/v25_operational_preds*.json",
    ], required=True, label="base preds v27/v25")
    rows = load_json(p)
    print("Loaded base rows:", len(rows))
    return rows, p

def load_candidates(required=True):
    p = resolve_one([
        "/kaggle/input/**/v23_val_candidates.json",
        "/kaggle/input/**/v23*candidates*.json",
        "/kaggle/working/**/v23_val_candidates.json",
        "/kaggle/working/**/v23*candidates*.json",
    ], required=required, label="v23 candidates")
    if p is None: return None, None
    rows = load_json(p)
    print("Loaded candidate rows:", len(rows))
    return rows, p

def cand_map(candidates):
    out = {}
    for i,r in enumerate(candidates or []):
        idx = r.get("idx", r.get("id", r.get("record_idx", i)))
        out[int(idx)] = r
    return out

def assert_mc_frozen(base, rows):
    bad = []
    for b,r in zip(base, rows):
        if bool(b.get("is_mc")) and norm_answer(b.get("pred")) != norm_answer(r.get("pred")):
            bad.append((b.get("idx"), b.get("pred"), r.get("pred")))
    if bad:
        print("MC freeze violations:", bad[:20])
        raise AssertionError(f"MC freeze violated in {len(bad)} rows")
    print("MC freeze check: PASS")

def c_answer(c):
    return norm_answer(c.get("answer") or c.get("pred") or c.get("final_answer"))

def cand_text(c):
    return str(c.get("text") or c.get("raw_text") or c.get("explanation") or "")

def explicit_final_answer(c):
    txt = cand_text(c)
    m = re.search(r"\bFinal Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b", txt, flags=re.I)
    return norm_answer(m.group(1)) if m else "UNPARSEABLE"

def get_supports(c):
    supp = c.get("supporting_premises") or c.get("supporting") or []
    if isinstance(supp, str):
        nums = re.findall(r"\d+", supp)
        return [int(x) for x in nums]
    try:
        return list(supp)
    except Exception:
        return []

def get_cands(row, cmap):
    return (cmap.get(int(row.get("idx", -1))) or {}).get("candidates", [])

def row_candidate_counts(row, cmap):
    ans = [c_answer(c) for c in get_cands(row, cmap)]
    return Counter(a for a in ans if a in LABELS)

def build_candidate_table(base_rows, cmap):
    table = []
    for i,row in enumerate(base_rows):
        if bool(row.get("is_mc")):
            continue
        gold = norm_answer(row.get("gold"))
        base_pred = norm_answer(row.get("pred"))
        cands = get_cands(row, cmap)
        counts = Counter(c_answer(c) for c in cands)
        total = max(1, sum(counts.values()))
        for rank,c in enumerate(cands):
            ans = c_answer(c)
            if ans not in YNU_LABELS:
                continue
            txt = cand_text(c)
            supp = get_supports(c)
            final = explicit_final_answer(c)
            feats = {
                "rank": rank,
                "is_first": 1 if rank == 0 else 0,
                "ans_yes": 1 if ans == "Yes" else 0,
                "ans_no": 1 if ans == "No" else 0,
                "ans_unknown": 1 if ans == "Unknown" else 0,
                "base_same": 1 if ans == base_pred else 0,
                "vote_count": counts.get(ans, 0),
                "vote_share": counts.get(ans, 0)/total,
                "supp_len": len(supp),
                "has_supp": 1 if len(supp) > 0 else 0,
                "short_supp": 1 if 1 <= len(supp) <= 5 else 0,
                "long_supp": 1 if len(supp) > 8 else 0,
                "text_len": min(len(txt), 3000),
                "final_match": 1 if final == ans else 0,
                "final_yes": 1 if final == "Yes" else 0,
                "final_no": 1 if final == "No" else 0,
                "final_unknown": 1 if final == "Unknown" else 0,
                "has_contra_kw": 1 if re.search(r"\b(however|contradict|false|cannot conclude|not enough|unknown|not supported)\b", txt, flags=re.I) else 0,
                "has_true_kw": 1 if re.search(r"\b(true|therefore|thus|supported|logically follows)\b", txt, flags=re.I) else 0,
                "question_len": min(len(str(row.get("question",""))), 1000),
            }
            table.append({
                "row_i": i, "idx": row.get("idx", i), "gold": gold, "base_pred": base_pred,
                "answer": ans, "target": 1 if ans == gold else 0, "features": feats,
                "candidate": c,
            })
    return table

FEATURE_NAMES = [
    "rank","is_first","ans_yes","ans_no","ans_unknown","base_same",
    "vote_count","vote_share","supp_len","has_supp","short_supp","long_supp",
    "text_len","final_match","final_yes","final_no","final_unknown",
    "has_contra_kw","has_true_kw","question_len"
]

def Xy_from_table(table, row_filter=None):
    rows = [r for r in table if row_filter is None or r["row_i"] in row_filter]
    X = [[float(r["features"].get(f,0.0)) for f in FEATURE_NAMES] for r in rows]
    y = [int(r["target"]) for r in rows]
    return X, y, rows

def fit_model(X, y, kind="logreg"):
    try:
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        if kind == "logreg":
            from sklearn.linear_model import LogisticRegression
            model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", C=0.5, random_state=SEED))
        elif kind == "rf":
            from sklearn.ensemble import RandomForestClassifier
            model = RandomForestClassifier(n_estimators=150, max_depth=3, min_samples_leaf=4, class_weight="balanced", random_state=SEED)
        else:
            from sklearn.ensemble import GradientBoostingClassifier
            model = GradientBoostingClassifier(random_state=SEED, max_depth=2, n_estimators=60)
        model.fit(X, y)
        return model, "sklearn_" + kind
    except Exception as e:
        print("sklearn unavailable or fit failed:", repr(e))
        return None, "fallback_score"

def fallback_score(feats):
    score = 0.0
    score += 0.6*feats.get("base_same",0)
    score += 0.4*feats.get("vote_share",0)
    score += 0.2*feats.get("short_supp",0)
    score += 0.3*feats.get("final_match",0)
    score -= 0.25*feats.get("long_supp",0)
    score -= 0.25*feats.get("has_contra_kw",0)
    if feats.get("ans_no",0): score -= 0.15
    if feats.get("ans_unknown",0): score -= 0.10
    return score

def predict_scores(model, rows):
    if model is None:
        return [fallback_score(r["features"]) for r in rows]
    X = [[float(r["features"].get(f,0.0)) for f in FEATURE_NAMES] for r in rows]
    try:
        if hasattr(model, "predict_proba"):
            return [float(p[1]) for p in model.predict_proba(X)]
        return [float(x) for x in model.decision_function(X)]
    except Exception:
        return [fallback_score(r["features"]) for r in rows]

def build_preds_from_calibrator(base_rows, table, model=None, restrict_rows=None, score_bias=None):
    score_bias = score_bias or {"Yes":0.0, "No":0.0, "Unknown":0.0}
    by_row = defaultdict(list)
    rows_for_score = table
    scores = predict_scores(model, rows_for_score)
    for r,s in zip(rows_for_score, scores):
        ss = s + score_bias.get(r["answer"], 0.0)
        by_row[r["row_i"]].append((ss, r))
    out = []
    restrict_rows = set(restrict_rows) if restrict_rows is not None else None
    for i,row in enumerate(base_rows):
        rr = dict(row)
        if bool(row.get("is_mc")):
            rr["pred"] = norm_answer(row.get("pred"))
            rr["source"] = "mc_frozen_from_base"
        elif restrict_rows is not None and i not in restrict_rows:
            rr["pred"] = norm_answer(row.get("pred"))
            rr["source"] = "ynu_base_not_in_eval_subset"
        elif by_row.get(i):
            best = max(by_row[i], key=lambda x: x[0])[1]
            rr["pred"] = best["answer"]
            rr["source"] = "ynu_candidate_calibrator_v29"
        else:
            rr["pred"] = norm_answer(row.get("pred"))
            rr["source"] = "ynu_fallback_base_no_candidate"
        out.append(rr)
    assert_mc_frozen(base_rows, out)
    return out

def subset_rows(rows, idxs):
    s = set(idxs)
    return [r for i,r in enumerate(rows) if i in s]

def print_fit_status(train_m, hold_m, cv_report=None, base_m=None, oracle_m=None):
    gap = train_m["macro_f1"] - hold_m["macro_f1"]
    status = []
    if gap > 0.08: status.append("OVERFIT_RISK_HIGH")
    elif gap > 0.04: status.append("OVERFIT_RISK_MEDIUM")
    else: status.append("NO_STRONG_OVERFIT_SIGNAL")
    if oracle_m and oracle_m["macro_f1"] - hold_m["macro_f1"] > 0.08:
        status.append("UNDERFIT_OR_SELECTION_GAP")
    if base_m and hold_m["macro_f1"] < base_m["macro_f1"] - 0.04:
        status.append("WORSE_THAN_BASELINE_ON_HOLDOUT")
    if cv_report:
        status.append(cv_report.get("overfit_flag",""))
        status.append(cv_report.get("stability_flag",""))
    print("="*100)
    print("FIT DIAGNOSTIC")
    print("="*100)
    print("train_macro:", train_m["macro_f1"])
    print("holdout_macro:", hold_m["macro_f1"])
    print("gap:", gap)
    print("status:", [x for x in status if x])
    return status

In [ ]:
base_rows, base_path = load_base_preds()
base_metric = compute_metrics(base_rows, "v29_full_base")
print_metric(base_metric)

std_path = resolve_one(["/kaggle/input/**/v29_standard_summary.json", "/kaggle/working/**/v29_standard_summary.json"], required=False, label="v29 standard summary")
if std_path:
    std = load_json(std_path)
    print("Loaded standard summary:", std_path)
    print("selected_source:", std.get("selected_source"))
    if std.get("selected_source") == "v29_calibrator_selected":
        # For simplicity, prefer saved preds if present; otherwise rollback.
        pred_path = resolve_one(["/kaggle/input/**/v29_standard_preds.json", "/kaggle/working/**/v29_standard_preds.json"], required=False, label="v29 standard preds")
        if pred_path:
            rows = load_json(pred_path)
            source = "loaded_v29_standard_preds"
        else:
            rows = base_rows
            source = "no_v29_preds_rollback_to_base"
    else:
        rows = base_rows
        source = "standard_summary_rollback_to_base"
else:
    rows = base_rows
    source = "no_standard_summary_rollback_to_base"

assert_mc_frozen(base_rows, rows)
metric = compute_metrics(rows, "v29_full_operational")
print("Source:", source)
print_metric(metric)
summary = dict(version="v29_full_next_session", base_path=str(base_path), source=source, metric=metric)
save_json(summary, OUT_DIR / "v29_full_summary.json")
save_json(rows, OUT_DIR / "v29_full_preds.json")